# Comparing Two Medical Risk Classifiers

## Logistic Regression vs Random Forest for Chronic Kidney Disease Risk Prediction

### Project Objective

The goal of this project is to train and compare two different machine learning classifiers for **Chronic Kidney Disease (CKD) risk prediction**. The models are evaluated using clinical classification metrics, with a primary focus on **Recall** because missing a CKD-positive patient can be more concerning in a medical screening context.

### Dataset

    Dataset: UCI Risk Factor Prediction of Chronic Kidney Disease
    Dataset ID: 857
    Dataset Size: 200 patients
    Target Variable: class
**Classes:**

    1. ckd = 1
    2. notckd = 0

### Models

1. **Logistic Regression**
 interpretable linear classification baseline.
2. **Random Forest**
 nonlinear ensemble classification model.

### Data Preprocessing

* Removed metadata rows from the original dataset.
* Converted the target into binary labels.
* Checked missing values and duplicate records.
* Identified and excluded potential target leakage features:

  * **affected**
  * **stage**
  * **grf**
* Applied **One-Hot Encoding** to categorical features.
* Used the same preprocessing pipeline for both models.
* Used a **stratified 80/20 train-test split**.

### Evaluation Metrics

The models are evaluated using:

* Accuracy
* Precision
* Recall
* F1-Score
* Confusion Matrix
* False Positives
* False Negatives
* False Negative Rate
* 5-Fold Stratified Cross-Validation

### Clinical Evaluation Focus

**Primary Metric: Recall**

Recall is prioritized because a **False Negative** represents a CKD-positive patient who was incorrectly classified as not having CKD. Reducing missed CKD cases is therefore important for this comparison.

### Main Results

**Test Set:**

* Logistic Regression: 100% Accuracy, Precision, Recall, and F1-Score
* Random Forest: 100% Accuracy, Precision, Recall, and F1-Score
* Both models produced 0 False Positives and 0 False Negatives.

**5-Fold Cross-Validation:**

* Logistic Regression: 100% Accuracy, Precision, Recall, and F1-Score
* Random Forest: 99.00% Accuracy, 100% Precision, 98.46% Recall, and 99.22% F1-Score

### Final Recommendation

Based on the cross-validation results and the primary clinical metric of Recall, **Logistic Regression is the preferred model for this experiment**. It achieved perfect mean performance across all evaluated metrics with zero variation across the five folds.

### Important Limitation

The dataset contains only **200 patients**, so the perfect model performance should not be interpreted as proof of clinical reliability or production readiness. Larger datasets, independent external validation, and clinical testing would be required before deployment in a real healthcare environment.


## Step 0 Libraries

In [2]:


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    accuracy_score
)

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


## Step 1  Load the UCI Dataset

In [3]:


url = "https://archive.ics.uci.edu/static/public/857/risk+factor+prediction+of+chronic+kidney+disease.zip"

!wget -q "$url" -O kidney_dataset.zip

!unzip -o kidney_dataset.zip

Archive:  kidney_dataset.zip
  inflating: ckd-dataset-v2.csv      


## Step 2  Load and inspect the dataset

In [4]:
df = pd.read_csv("ckd-dataset-v2.csv")

print("Dataset Shape:", df.shape)

Dataset Shape: (202, 29)


In [5]:
print("\nFirst 5 Rows:")
display(df.head())


First 5 Rows:


,bp (Diastolic),bp limit,sg,al,class,rbc,su,pc,pcc,ba,...,htn,dm,cad,appet,pe,ane,grf,stage,affected,age
0,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,...,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,class,meta
2,0,0,1.019 - 1.021,1 - 1,ckd,0,< 0,0,0,0,...,0,0,0,0,0,0,≥ 227.944,s1,1,< 12
3,0,0,1.009 - 1.011,< 0,ckd,0,< 0,0,0,0,...,0,0,0,0,0,0,≥ 227.944,s1,1,< 12
4,0,0,1.009 - 1.011,≥ 4,ckd,1,< 0,1,0,1,...,0,0,0,1,0,0,127.281 - 152.446,s1,1,< 12


In [6]:
print("\nColumn Names:")
print(df.columns.tolist())


Column Names:
['bp (Diastolic)', 'bp limit', 'sg', 'al', 'class', 'rbc', 'su', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sod', 'sc', 'pot', 'hemo', 'pcv', 'rbcc', 'wbcc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'grf', 'stage', 'affected', 'age']


In [7]:
print("\nData Types:")
display(df.dtypes)


Data Types:


,0
bp (Diastolic),object
bp limit,object
sg,object
al,object
class,object
rbc,object
su,object
pc,object
pcc,object
ba,object


In [8]:
print("\nDataset Information:")
df.info()


Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 202 entries, 0 to 201
Data columns (total 29 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   bp (Diastolic)  201 non-null    object
 1   bp limit        201 non-null    object
 2   sg              201 non-null    object
 3   al              201 non-null    object
 4   class           201 non-null    object
 5   rbc             201 non-null    object
 6   su              201 non-null    object
 7   pc              201 non-null    object
 8   pcc             201 non-null    object
 9   ba              201 non-null    object
 10  bgr             201 non-null    object
 11  bu              201 non-null    object
 12  sod             201 non-null    object
 13  sc              201 non-null    object
 14  pot             201 non-null    object
 15  hemo            201 non-null    object
 16  pcv             201 non-null    object
 17  rbcc            201 non-null    

In [9]:
# Basic statistical overview

print("Numerical Summary:")
display(df.describe())

print("\nCategorical Summary:")
display(df.describe(include="object"))

Numerical Summary:


,bp (Diastolic),bp limit,sg,al,class,rbc,su,pc,pcc,ba,...,htn,dm,cad,appet,pe,ane,grf,stage,affected,age
count,201,201,201,201,201,201,201,201,201,201,...,201,201,201,201,201,201,201,201,202,202
unique,3,4,6,6,3,3,7,3,3,3,...,3,3,3,3,3,3,12,6,4,12
top,1,0,1.019 - 1.021,< 0,ckd,0,< 0,0,0,0,...,0,0,0,0,0,0,< 26.6175,s1,1,59 - 66
freq,108,95,75,116,128,175,170,155,173,189,...,122,130,178,160,165,168,68,54,128,48



Categorical Summary:


,bp (Diastolic),bp limit,sg,al,class,rbc,su,pc,pcc,ba,...,htn,dm,cad,appet,pe,ane,grf,stage,affected,age
count,201,201,201,201,201,201,201,201,201,201,...,201,201,201,201,201,201,201,201,202,202
unique,3,4,6,6,3,3,7,3,3,3,...,3,3,3,3,3,3,12,6,4,12
top,1,0,1.019 - 1.021,< 0,ckd,0,< 0,0,0,0,...,0,0,0,0,0,0,< 26.6175,s1,1,59 - 66
freq,108,95,75,116,128,175,170,155,173,189,...,122,130,178,160,165,168,68,54,128,48


In [10]:


print("First 5 rows:")
display(df.head())

print("\nUnique values in target column:")
print(df["class"].unique())

print("\nValue counts of target:")
print(df["class"].value_counts(dropna=False))

First 5 rows:


,bp (Diastolic),bp limit,sg,al,class,rbc,su,pc,pcc,ba,...,htn,dm,cad,appet,pe,ane,grf,stage,affected,age
0,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,...,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,class,meta
2,0,0,1.019 - 1.021,1 - 1,ckd,0,< 0,0,0,0,...,0,0,0,0,0,0,≥ 227.944,s1,1,< 12
3,0,0,1.009 - 1.011,< 0,ckd,0,< 0,0,0,0,...,0,0,0,0,0,0,≥ 227.944,s1,1,< 12
4,0,0,1.009 - 1.011,≥ 4,ckd,1,< 0,1,0,1,...,0,0,0,1,0,0,127.281 - 152.446,s1,1,< 12



Unique values in target column:
['discrete' nan 'ckd' 'notckd']

Value counts of target:
class
ckd         128
notckd       72
NaN           1
discrete      1
Name: count, dtype: int64


In [11]:
# Inspect rows that contain metadata instead of patient records

print("Rows containing 'discrete':")
display(df[df.astype(str).apply(
    lambda row: row.str.contains("discrete", case=False, na=False).any(),
    axis=1
)])

print("\nRows containing 'meta':")
display(df[df.astype(str).apply(
    lambda row: row.str.contains("meta", case=False, na=False).any(),
    axis=1
)])

Rows containing 'discrete':


,bp (Diastolic),bp limit,sg,al,class,rbc,su,pc,pcc,ba,...,htn,dm,cad,appet,pe,ane,grf,stage,affected,age
0,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,...,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete,discrete



Rows containing 'meta':


,bp (Diastolic),bp limit,sg,al,class,rbc,su,pc,pcc,ba,...,htn,dm,cad,appet,pe,ane,grf,stage,affected,age
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,class,meta


In [12]:
# Inspect the last few rows as well

display(df.tail())

,bp (Diastolic),bp limit,sg,al,class,rbc,su,pc,pcc,ba,...,htn,dm,cad,appet,pe,ane,grf,stage,affected,age
197,1,2,1.019 - 1.021,< 0,ckd,0,< 0,0,0,0,...,1,1,0,0,0,1,26.6175 - 51.7832,s3,1,≥ 74
198,0,0,1.019 - 1.021,< 0,ckd,0,< 0,0,0,0,...,0,1,0,0,0,1,< 26.6175,s4,1,≥ 74
199,1,1,≥ 1.023,< 0,notckd,0,< 0,0,0,0,...,0,0,0,0,0,0,51.7832 - 76.949,s2,0,≥ 74
200,1,1,≥ 1.023,< 0,notckd,0,< 0,0,0,0,...,0,0,0,0,0,0,102.115 - 127.281,s1,0,≥ 74
201,1,1,1.009 - 1.011,2 - 2,ckd,0,2 - 2,0,0,0,...,1,1,0,0,0,0,< 26.6175,s4,1,≥ 74


## Step 3  Data Quality & Target Analysis

###Step 3.1 Remove metadata rows

In [13]:


df = df[df["class"].isin(["ckd", "notckd"])].copy()

print("Cleaned Dataset Shape:", df.shape)

print("\nTarget Distribution:")
print(df["class"].value_counts())

print("\nTarget Distribution (%):")
print((df["class"].value_counts(normalize=True) * 100).round(2))

Cleaned Dataset Shape: (200, 29)

Target Distribution:
class
ckd       128
notckd     72
Name: count, dtype: int64

Target Distribution (%):
class
ckd       64.0
notckd    36.0
Name: proportion, dtype: float64


### 3.2 Convert target in 0/1 form

In [14]:


df["target"] = df["class"].map({
    "notckd": 0,
    "ckd": 1
})

print(df[["class", "target"]].value_counts())

print("\nUnique target values:")
print(df["target"].unique())

print("\nMissing target values:")
print(df["target"].isna().sum())

class   target
ckd     1         128
notckd  0          72
Name: count, dtype: int64

Unique target values:
[1 0]

Missing target values:
0


### 3.3 Missing values check

In [15]:


missing_values = df.isnull().sum()

print("Missing Values:")
print(missing_values[missing_values > 0])

Missing Values:
Series([], dtype: int64)


### 3.4 Duplicate rows check

In [16]:


print("Number of duplicate rows:", df.duplicated().sum())

Number of duplicate rows: 0


### 3.5 Potential Data Leakage Check

In [17]:


potential_risk_features = [
    "grf",
    "stage",
    "affected",
    "htn",
    "dm",
    "cad"
]

for col in potential_risk_features:
    print(f"\n--- {col} ---")
    print(
        pd.crosstab(
            df[col],
            df["class"],
            normalize="columns"
        ).round(3)
    )


--- grf ---
class                ckd  notckd
grf                             
 p                 0.008   0.000
102.115 - 127.281  0.023   0.167
127.281 - 152.446  0.023   0.111
152.446 - 177.612  0.000   0.125
177.612 - 202.778  0.008   0.097
202.778 - 227.944  0.000   0.042
26.6175 - 51.7832  0.281   0.028
51.7832 - 76.949   0.094   0.222
76.949 - 102.115   0.031   0.181
< 26.6175          0.516   0.028
≥ 227.944          0.016   0.000

--- stage ---
class    ckd  notckd
stage               
s1     0.070   0.625
s2     0.094   0.319
s3     0.242   0.000
s4     0.320   0.056
s5     0.273   0.000

--- affected ---
class     ckd  notckd
affected             
0         0.0     1.0
1         1.0     0.0

--- htn ---
class    ckd  notckd
htn                 
0      0.391     1.0
1      0.609     0.0

--- dm ---
class    ckd  notckd
dm                  
0      0.453     1.0
1      0.547     0.0

--- cad ---
class    ckd  notckd
cad                 
0      0.828     1.0
1      0.172     0.0


### Step 3.6 Inspect unique values of each feature

In [18]:


for col in df.columns:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False).head(15))


bp (Diastolic)
bp (Diastolic)
1    108
0     92
Name: count, dtype: int64

bp limit
bp limit
0    95
1    59
2    46
Name: count, dtype: int64

sg
sg
1.019 - 1.021    75
1.009 - 1.011    45
≥ 1.023          41
1.015 - 1.017    36
< 1.007           3
Name: count, dtype: int64

al
al
< 0      116
2 - 2     27
3 - 3     23
1 - 1     21
≥ 4       13
Name: count, dtype: int64

class
class
ckd       128
notckd     72
Name: count, dtype: int64

rbc
rbc
0    175
1     25
Name: count, dtype: int64

su
su
< 0      170
2 - 2      9
3 - 4      8
4 - 4      6
1 - 2      6
≥ 4        1
Name: count, dtype: int64

pc
pc
0    155
1     45
Name: count, dtype: int64

pcc
pcc
0    173
1     27
Name: count, dtype: int64

ba
ba
0    189
1     11
Name: count, dtype: int64

bgr
bgr
112 - 154    79
< 112        70
196 - 238    14
154 - 196    13
238 - 280    11
406 - 448     4
280 - 322     4
≥ 448         3
364 - 406     1
322 - 364     1
Name: count, dtype: int64

bu
bu
< 48.1           108
48.1 - 86.2     

## Step 3.7 Feature Policy

### Objective

The goal of this milestone is to compare medical risk classifiers.

We should train the model on meaningful risk factors instead of features that directly reveal the diagnosis.

### Remove

The following feature should be removed because it can directly reveal the diagnosis:

- `affected`

### Also Exclude

For a more conservative and realistic prospective-risk setup, we will also exclude:

- `stage`
- `grf`

This makes the model's prediction task more realistic for early risk assessment.

### Keep

We will keep the remaining clinically relevant features:

- `bp`  Diastolic blood pressure
- `bp limit`
- `sg`
- `al`
- `rbc`
- `su`
- `pc`
- `pcc`
- `ba`
- `bgr`
- `bu`
- `sod`
- `sc`
- `pot`
- `hemo`
- `pcv`
- `rbcc`
- `wbcc`
- `htn`
- `dm`
- `cad`
- `appet`
- `pe`
- `ane`
- `age`

### Important Note

`htn`, `dm`, and `cad` are not automatically considered leakage.

They represent comorbidities or risk factors and can be relevant for prospective medical risk assessment.

Therefore, we will keep them in the feature set.

### Step 3.8  Final Leakage Check

In [19]:


leakage_features = ["affected", "stage", "grf"]

df_model = df.drop(columns=leakage_features)

print("Original shape:", df.shape)
print("Modeling dataset shape:", df_model.shape)

print("\nRemoved features:")
print(leakage_features)

print("\nRemaining features:")
print(df_model.columns.tolist())

Original shape: (200, 30)
Modeling dataset shape: (200, 27)

Removed features:
['affected', 'stage', 'grf']

Remaining features:
['bp (Diastolic)', 'bp limit', 'sg', 'al', 'class', 'rbc', 'su', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sod', 'sc', 'pot', 'hemo', 'pcv', 'rbcc', 'wbcc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'age', 'target']


### Step 3.9  Separate X and y

In [20]:


X = df_model.drop(columns=["class", "target"])
y = df_model["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

X shape: (200, 25)
y shape: (200,)

Target distribution:
target
1    128
0     72
Name: count, dtype: int64


## Step 4  Train/Test Split

## Why do we split the data?

We need to separate the dataset into two parts:

- **Training set:** Used by the model to learn patterns from the data.
- **Test set:** Completely unseen data used for the final evaluation.

Because our dataset is small (200 patients) and the target classes are imbalanced, we will use:

- **80% training data**
- **20% testing data**
- **stratify=y**   keeps a similar CKD/non-CKD ratio in both sets.
- **random_state=42**    makes the split reproducible, so we get the same result every time.

This helps us evaluate the model fairly on unseen medical data.

In [21]:


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

print("\nTraining target distribution (%):")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTesting target distribution (%):")
print((y_test.value_counts(normalize=True) * 100).round(2))

Training set shape: (160, 25)
Testing set shape: (40, 25)

Training target distribution:
target
1    102
0     58
Name: count, dtype: int64

Testing target distribution:
target
1    26
0    14
Name: count, dtype: int64

Training target distribution (%):
target
1    63.75
0    36.25
Name: proportion, dtype: float64

Testing target distribution (%):
target
1    65.0
0    35.0
Name: proportion, dtype: float64


## Step 5  Preprocessing Pipeline

## Why this step?

Our 25 features are currently stored as object values.

Many of these features contain categorical or range-based values, for example:

1.  **sg**  1.015-1.020
2.  **hemo**  10-12
3. **bgr**  120-140
4.  **age**  50-59

We should **not simply convert these strings into numbers**, because values such as 10-12 represent a category/range, not a single numeric value.

For this milestone, we will treat these values as **categorical features** and use:

**OneHotEncoder**

The encoder converts each category into machine-readable binary columns.

### Important: Prevent Data Leakage

Preprocessing will be included **inside the model pipelines**.

This ensures that the encoder learns categories only from the training data and does not use information from the test set.

### Pipeline Structure

**Logistic Regression**

```text
Training Data
      ↓
OneHotEncoder
      ↓
Logistic Regression

Random Forest

Training Data
      ↓
OneHotEncoder
      ↓
Random Forest

Both classifiers will use the same preprocessing approach, making the model comparison fair and consistent.

### Step 5.1  Identify categorical features

In [22]:
# Check feature data types
print("Feature data types:")
print(X.dtypes)

# Separate categorical features
categorical_features = X.columns.tolist()

print("\nNumber of categorical features:", len(categorical_features))
print("\nCategorical features:")
print(categorical_features)

Feature data types:
bp (Diastolic)    object
bp limit          object
sg                object
al                object
rbc               object
su                object
pc                object
pcc               object
ba                object
bgr               object
bu                object
sod               object
sc                object
pot               object
hemo              object
pcv               object
rbcc              object
wbcc              object
htn               object
dm                object
cad               object
appet             object
pe                object
ane               object
age               object
dtype: object

Number of categorical features: 25

Categorical features:
['bp (Diastolic)', 'bp limit', 'sg', 'al', 'rbc', 'su', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sod', 'sc', 'pot', 'hemo', 'pcv', 'rbcc', 'wbcc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'age']


### Step 5.2  Preprocessing pipeline

In [23]:


categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")
print("Categorical features:", len(categorical_features))

Preprocessing pipeline created successfully.
Categorical features: 25


##Step 6 Model 1 Logistic Regression Pipeline

In [24]:


logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

# Train the model
logistic_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


## Step 7  Model 2 Random Forest

Now we train the second classifier using the same training data and same preprocessing pipeline. This makes the comparison fair.

**Concept**

Random Forest is an ensemble of multiple decision trees. It can capture nonlinear relationships and interactions between medical risk factors.

We will use it as a contrasting model to Logistic Regression.

In [25]:


random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            random_state=42
        ))
    ]
)

# Train the model
random_forest_model.fit(X_train, y_train)

print("Random Forest model trained successfully.")

Random Forest model trained successfully.


## Step 8  Generate Predictions

Now we use both trained models on the same unseen test set.

This is important because we want an identical basis for comparison.

In [26]:


y_pred_logistic = logistic_model.predict(X_test)
y_pred_rf = random_forest_model.predict(X_test)

print("Predictions generated successfully.")

print("\nLogistic Regression predictions:")
print(pd.Series(y_pred_logistic).value_counts().sort_index())

print("\nRandom Forest predictions:")
print(pd.Series(y_pred_rf).value_counts().sort_index())

Predictions generated successfully.

Logistic Regression predictions:
0    14
1    26
Name: count, dtype: int64

Random Forest predictions:
0    14
1    26
Name: count, dtype: int64


## Step 9  Confusion Matrices

We will use the confusion matrix to understand how well the model predicts CKD and non-CKD.

The four values are:

1.  **TN (True Negative)**  Correctly predicted **Not CKD**
2. **FP (False Positive)**   Predicted **CKD**, but the patient is actually **Not CKD**
3.  **FN (False Negative)**  Predicted **Not CKD**, but the patient actually has **CKD**
4. **TP (True Positive)**  Correctly predicted **CKD**

### Clinical Importance

For our medical risk classification task, **FN is especially important**.

A false negative means that a patient with CKD was not identified by the model.

Therefore, we will pay particular attention to:

- **FN**
- **Recall**
- **Sensitivity**

A lower number of false negatives means the model is missing fewer CKD patients.

In [27]:


cm_logistic = confusion_matrix(y_test, y_pred_logistic)
cm_rf = confusion_matrix(y_test, y_pred_rf)

print("Logistic Regression Confusion Matrix:")
print(cm_logistic)

print("\nRandom Forest Confusion Matrix:")
print(cm_rf)

Logistic Regression Confusion Matrix:
[[14  0]
 [ 0 26]]

Random Forest Confusion Matrix:
[[14  0]
 [ 0 26]]


## Step 10 Precision, Recall and F1

In [28]:


logistic_precision = precision_score(y_test, y_pred_logistic, pos_label=1)
logistic_recall = recall_score(y_test, y_pred_logistic, pos_label=1)
logistic_f1 = f1_score(y_test, y_pred_logistic, pos_label=1)

rf_precision = precision_score(y_test, y_pred_rf, pos_label=1)
rf_recall = recall_score(y_test, y_pred_rf, pos_label=1)
rf_f1 = f1_score(y_test, y_pred_rf, pos_label=1)

print("Logistic Regression")
print(f"Precision: {logistic_precision:.4f}")
print(f"Recall:    {logistic_recall:.4f}")
print(f"F1-score:  {logistic_f1:.4f}")

print("\nRandom Forest")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall:    {rf_recall:.4f}")
print(f"F1-score:  {rf_f1:.4f}")

Logistic Regression
Precision: 1.0000
Recall:    1.0000
F1-score:  1.0000

Random Forest
Precision: 1.0000
Recall:    1.0000
F1-score:  1.0000


### Clinical Interpretation

For the **40-patient test set**:

1.  **Recall = 1.0**  Both models detected all **26 CKD patients**.
2.  **False Negatives = 0**  Neither model missed a CKD case.
3.  **Precision = 1.0**  Every patient predicted as CKD actually had CKD.
4.  **F1-Score = 1.0** Both models achieved a perfect balance between Precision and Recall on this test set.

### Important Note

This does **not** mean that the models are production-ready.

The test set contains only **40 patients**, so perfect performance may occur by chance and may not generalize to new patients.

A **larger and independent validation dataset** would be needed before considering clinical use.

## Step 11  Side-by-Side Comparison

In [29]:


comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Precision": [logistic_precision, rf_precision],
    "Recall": [logistic_recall, rf_recall],
    "F1-Score": [logistic_f1, rf_f1],
    "False Positives": [cm_logistic[0, 1], cm_rf[0, 1]],
    "False Negatives": [cm_logistic[1, 0], cm_rf[1, 0]]
})

comparison

,Model,Precision,Recall,F1-Score,False Positives,False Negatives
0,Logistic Regression,1.0,1.0,1.0,0,0
1,Random Forest,1.0,1.0,1.0,0,0


## Step 12  Recall-Focused Clinical Analysis

Because our main clinical objective is to prioritize **Recall**, we will analyze the confusion matrix with a focus on missed CKD cases.

### Recall Formula

> **Recall = TP / (TP + FN)**

For both models:

- **TP = 26**
- **FN = 0**
- **Recall = 26 / (26 + 0) = 1.00**
- **Missed CKD cases = 0**

This means both models identified all CKD patients in our test set.

### False Negative Rate (FNR)

We will also calculate the **False Negative Rate (FNR)** because it shows the proportion of actual CKD patients that the model missed.

> **FNR = FN / (TP + FN)**

For both models:

- **FN = 0**
- **TP = 26**
- **FNR = 0 / (26 + 0) = 0.00**

Therefore, both models had a **0% False Negative Rate** on this test set.

### Clinical Interpretation

1. **Higher Recall** fewer CKD patients are missed.
2. **Lower FNR**  fewer CKD patients are missed.
3. Both models achieved **Recall = 1.00** and **FNR = 0.00** on this 40-patient test set.

In [30]:


logistic_fnr = cm_logistic[1, 0] / (cm_logistic[1, 0] + cm_logistic[1, 1])
rf_fnr = cm_rf[1, 0] / (cm_rf[1, 0] + cm_rf[1, 1])

print("Clinical Recall Analysis")
print("-" * 40)

print(f"Logistic Regression Recall: {logistic_recall:.2%}")
print(f"Logistic Regression False Negative Rate: {logistic_fnr:.2%}")
print(f"Logistic Regression Missed CKD Cases: {cm_logistic[1, 0]}")

print()

print(f"Random Forest Recall: {rf_recall:.2%}")
print(f"Random Forest False Negative Rate: {rf_fnr:.2%}")
print(f"Random Forest Missed CKD Cases: {cm_rf[1, 0]}")

Clinical Recall Analysis
----------------------------------------
Logistic Regression Recall: 100.00%
Logistic Regression False Negative Rate: 0.00%
Logistic Regression Missed CKD Cases: 0

Random Forest Recall: 100.00%
Random Forest False Negative Rate: 0.00%
Random Forest Missed CKD Cases: 0


## Step 13 Final Recommendation

In [31]:


print("Final Model Recommendation")
print("=" * 50)

if logistic_recall > rf_recall:
    recommendation = "Logistic Regression"
elif rf_recall > logistic_recall:
    recommendation = "Random Forest"
else:
    recommendation = "Tie"

print(f"Logistic Regression Recall: {logistic_recall:.2%}")
print(f"Random Forest Recall:       {rf_recall:.2%}")
print()

if recommendation == "Tie":
    print("Recommendation: Both models performed equally on the test set.")
    print("Both achieved 100% Recall with 0 missed CKD cases.")
else:
    print(f"Recommended Model: {recommendation}")

Final Model Recommendation
Logistic Regression Recall: 100.00%
Random Forest Recall:       100.00%

Recommendation: Both models performed equally on the test set.
Both achieved 100% Recall with 0 missed CKD cases.


In [32]:
# Check training vs test performance

y_train_pred_logistic = logistic_model.predict(X_train)
y_train_pred_rf = random_forest_model.predict(X_train)

train_acc_logistic = accuracy_score(y_train, y_train_pred_logistic)
test_acc_logistic = accuracy_score(y_test, y_pred_logistic)

train_acc_rf = accuracy_score(y_train, y_train_pred_rf)
test_acc_rf = accuracy_score(y_test, y_pred_rf)

print("Training vs Test Accuracy")
print("=" * 40)

print(f"Logistic Regression - Train: {train_acc_logistic:.2%}")
print(f"Logistic Regression - Test:  {test_acc_logistic:.2%}")

print()

print(f"Random Forest - Train:       {train_acc_rf:.2%}")
print(f"Random Forest - Test:        {test_acc_rf:.2%}")

Training vs Test Accuracy
Logistic Regression - Train: 100.00%
Logistic Regression - Test:  100.00%

Random Forest - Train:       100.00%
Random Forest - Test:        100.00%


Both models achieved 100% training and test accuracy with no train-test gap. However, due to the small dataset and perfect performance, generalization cannot be confirmed. Cross-validation is required before concluding that the models are robust.

## Step 14  5-Fold Cross-Validation

In [33]:


from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

# Logistic Regression CV
logistic_cv = cross_validate(
    logistic_model,
    X,
    y,
    cv=cv,
    scoring=scoring
)

# Random Forest CV
rf_cv = cross_validate(
    random_forest_model,
    X,
    y,
    cv=cv,
    scoring=scoring
)

print("5-Fold Cross-Validation Results")
print("=" * 50)

print("\nLogistic Regression")
print(f"Accuracy:  {logistic_cv['test_accuracy'].mean():.4f} ± {logistic_cv['test_accuracy'].std():.4f}")
print(f"Precision: {logistic_cv['test_precision'].mean():.4f} ± {logistic_cv['test_precision'].std():.4f}")
print(f"Recall:    {logistic_cv['test_recall'].mean():.4f} ± {logistic_cv['test_recall'].std():.4f}")
print(f"F1-Score:  {logistic_cv['test_f1'].mean():.4f} ± {logistic_cv['test_f1'].std():.4f}")

print("\nRandom Forest")
print(f"Accuracy:  {rf_cv['test_accuracy'].mean():.4f} ± {rf_cv['test_accuracy'].std():.4f}")
print(f"Precision: {rf_cv['test_precision'].mean():.4f} ± {rf_cv['test_precision'].std():.4f}")
print(f"Recall:    {rf_cv['test_recall'].mean():.4f} ± {rf_cv['test_recall'].std():.4f}")
print(f"F1-Score:  {rf_cv['test_f1'].mean():.4f} ± {rf_cv['test_f1'].std():.4f}")

5-Fold Cross-Validation Results

Logistic Regression
Accuracy:  1.0000 ± 0.0000
Precision: 1.0000 ± 0.0000
Recall:    1.0000 ± 0.0000
F1-Score:  1.0000 ± 0.0000

Random Forest
Accuracy:  0.9900 ± 0.0122
Precision: 1.0000 ± 0.0000
Recall:    0.9846 ± 0.0188
F1-Score:  0.9922 ± 0.0096


## Step 15. Conclusion

Two medical risk classifiers, Logistic Regression and Random Forest, were trained and evaluated on the same Chronic Kidney Disease dataset using the same preprocessing approach.

On the 40-patient test set, both models achieved 100% Accuracy, Precision, Recall, and F1-Score, with 0 False Positives and 0 False Negatives. Since missing a CKD case is clinically more concerning, Recall was treated as the primary metric.

Five-fold stratified cross-validation provided a broader evaluation. Logistic Regression achieved 100% mean Accuracy, Precision, Recall, and F1-Score with zero variation across folds. Random Forest achieved 99.00% Accuracy, 98.46% Recall, and 99.22% F1-Score. Therefore, Logistic Regression is the preferred model for this experiment because it showed more consistent performance and achieved the highest Recall.

However, the dataset contains only 200 patients, so the perfect performance should not be considered proof of clinical reliability or production readiness. Larger datasets, independent external validation, and clinical testing would be required before real-world use.

Overall, this project demonstrated how different medical classifiers can be compared using confusion matrices, Precision, Recall, F1-Score, and cross-validation, with particular focus on minimizing missed CKD cases.